# StageBridge: Receiver-Centered Niche Encoding for Cell State Transitions

This notebook demonstrates the StageBridge architecture and analyzes the trained model's learned representations. The central hypothesis is that cross-sectional cell state transitions become more identifiable when a focal receiver cell is modeled together with its local microenvironmental niche rather than as an isolated expression profile.

**Contents:**
1. Semi-synthetic benchmark with ground-truth interactions
2. Trained model weight analysis
3. Ablation study visualization
4. Conclusions and interpretation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import json
from pathlib import Path
from scipy.spatial.distance import cdist

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white',
})

RESULTS_DIR = Path('results/v1')
DATA_DIR = Path('data')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 1. Semi-Synthetic Benchmark

The semi-synthetic benchmark provides ground-truth interaction labels for rigorous evaluation. Following the AMICI methodology (Hong et al. 2025), spatial proximity between sender and receiver cells is explicitly controlled, enabling direct assessment of whether learned attention patterns align with known cell-cell communication structure.

Nine interaction rules are defined based on established ligand-receptor pairs from lung cancer biology, including IL1B-IL1R1 (macrophage to epithelial, 50μm range), CXCL12-CXCR4 (fibroblast to epithelial, 100μm range), and PDL1-PD1 (epithelial to T cell, 40μm range).

In [ ]:
# Load semi-synthetic benchmark
benchmark_size = 'small'  # Options: 'small' (1K cells), 'medium' (5K), 'large' (20K)
benchmark_dir = DATA_DIR / f'semisynthetic_benchmark_{benchmark_size}'

coords = pd.read_parquet(benchmark_dir / 'coordinates.parquet')
neighborhoods = pd.read_parquet(benchmark_dir / 'neighborhoods.parquet')
labels = pd.read_parquet(benchmark_dir / 'ground_truth_labels_fixed.parquet')

with open(benchmark_dir / 'summary.json') as f:
    summary = json.load(f)

print(f'Benchmark: {benchmark_size}')
print(f'Total cells: {len(coords)}')
print(f'Interacting: {labels["is_interacting"].sum()} ({100*labels["is_interacting"].mean():.1f}%)')
print(f'\nCell type distribution:')
print(coords['cell_type'].value_counts())

In [ ]:
# Spatial layout and ground-truth interactions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Cell types
ax = axes[0]
cell_type_colors = {
    'AT2': '#1f77b4', 'Basal': '#ff7f0e', 'Ciliated': '#2ca02c', 'Secretory': '#d62728',
    'Macrophages': '#9467bd', 'Fibroblast lineage': '#8c564b', 'T cell lineage': '#e377c2',
    'Capillary': '#7f7f7f', 'Mast cells': '#bcbd22'
}
for ct in coords['cell_type'].unique():
    mask = coords['cell_type'] == ct
    ax.scatter(coords.loc[mask, 'x'], coords.loc[mask, 'y'],
               c=cell_type_colors.get(ct, 'gray'), s=8, alpha=0.6, label=ct)
ax.set_xlabel('X (μm)')
ax.set_ylabel('Y (μm)')
ax.set_title('Cell Type Distribution')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)

# Interacting cells
ax = axes[1]
non_int = ~labels['is_interacting'].values
ax.scatter(coords.loc[non_int, 'x'], coords.loc[non_int, 'y'],
           c='#e0e0e0', s=6, alpha=0.4, label='Non-interacting')
ax.scatter(coords.loc[labels['is_interacting'].values, 'x'],
           coords.loc[labels['is_interacting'].values, 'y'],
           c='#c0392b', s=12, alpha=0.7, label='Interacting')
ax.set_xlabel('X (μm)')
ax.set_ylabel('Y (μm)')
ax.set_title('Ground-Truth Interactions')
ax.legend()

# Expected attention ring
ax = axes[2]
ring_counts = labels[labels['is_interacting']]['expected_attention_ring'].value_counts().sort_index()
ring_labels = ['Ring 1\n(0-50μm)', 'Ring 2\n(50-100μm)', 'Ring 3\n(100-150μm)', 'Ring 4\n(150-200μm)']
colors = ['#c0392b', '#e74c3c', '#f39c12', '#f1c40f']
bars = ax.bar(range(len(ring_counts)), ring_counts.values, color=colors[:len(ring_counts)])
ax.set_xticks(range(len(ring_counts)))
ax.set_xticklabels([ring_labels[i] for i in ring_counts.index])
ax.set_ylabel('Count')
ax.set_title('Expected Attention Ring\n(nearest sender location)')
for bar, val in zip(bars, ring_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            str(val), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('figures/publication/fig_semisynthetic_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Trained Model Analysis

The trained model checkpoint contains learned weights that reveal how the architecture processes niche information. The drift head implements a gated combination of two pathways: a latent-only path that predicts transitions from receiver cell state alone, and a context path that uses cross-attention over niche tokens. Comparing weight magnitudes between these paths indicates whether the model learned to rely on niche context.

In [ ]:
# Load trained model checkpoint
checkpoint_path = RESULTS_DIR / 'full/fold_1/seed_42/weights/final_model.pt'
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
state = checkpoint['model_state_dict']

print('Model configuration:')
for k, v in checkpoint['config']['model_config'].items():
    print(f'  {k}: {v}')

n_params = sum(p.numel() for p in state.values())
print(f'\nTotal parameters: {n_params:,}')

In [ ]:
# Drift head pathway analysis
# The drift head has two paths: latent-only (receiver + stage + time) and context (cross-attention)
latent_only_norm = state['drift_head.latent_only.0.weight'].norm().item()
context_out_norm = state['drift_head.context_out_proj.weight'].norm().item()
gate_bias = state['drift_head.context_gate.2.bias'].item()

print('Drift Head Pathway Analysis')
print('=' * 50)
print(f'Latent-only path weight norm: {latent_only_norm:.4f}')
print(f'Context path weight norm:     {context_out_norm:.4f}')
print(f'Ratio (latent/context):       {latent_only_norm/context_out_norm:.2f}x')
print(f'\nContext gate bias: {gate_bias:.4f}')
print(f'Default gate value: sigmoid({gate_bias:.4f}) = {torch.sigmoid(torch.tensor(gate_bias)).item():.4f}')

In [ ]:
# Token projection and ring differentiation analysis
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Panel A: Token projection weights
ax = axes[0, 0]
token_norms = {
    'Receiver': state['niche_tokenizer.token_proj.weight'].norm().item(),
    'HLCA': state['niche_tokenizer.hlca_proj.weight'].norm().item(),
    'LuCA': state['niche_tokenizer.luca_proj.weight'].norm().item(),
    'Stats': state['niche_tokenizer.stats_proj.weight'].norm().item(),
}
for i in range(4):
    token_norms[f'Ring {i+1}'] = state[f'niche_tokenizer.ring_poolers.{i}.proj.weight'].norm().item()

colors = ['#27ae60', '#2980b9', '#8e44ad', '#7f8c8d', '#c0392b', '#e74c3c', '#f39c12', '#f1c40f']
bars = ax.bar(token_norms.keys(), token_norms.values(), color=colors)
ax.set_ylabel('Projection Weight Norm')
ax.set_title('A. Token Input Projections')
ax.tick_params(axis='x', rotation=45)

# Panel B: Ring PMA seed similarity
ax = axes[0, 1]
seeds = torch.stack([state[f'niche_tokenizer.ring_poolers.{i}.pma.seed_vectors'].squeeze() for i in range(4)])
seeds_norm = seeds / seeds.norm(dim=1, keepdim=True)
sim = (seeds_norm @ seeds_norm.T).numpy()
sns.heatmap(sim, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=['Ring 1', 'Ring 2', 'Ring 3', 'Ring 4'],
            yticklabels=['Ring 1', 'Ring 2', 'Ring 3', 'Ring 4'],
            ax=ax, cbar_kws={'label': 'Cosine Similarity'})
ax.set_title('B. Ring PMA Seed Similarity')

# Panel C: Drift head pathway comparison
ax = axes[1, 0]
pathway_data = {
    'Latent-only\n(no context)': latent_only_norm,
    'Context\n(cross-attention)': context_out_norm,
}
bars = ax.bar(pathway_data.keys(), pathway_data.values(), color=['#27ae60', '#c0392b'])
ax.set_ylabel('Weight Norm')
ax.set_title('C. Drift Head Pathway Weights')
ax.axhline(y=0, color='black', linewidth=0.5)
for bar, val in zip(bars, pathway_data.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{val:.2f}', ha='center', fontsize=10)

# Panel D: Gate input weights
ax = axes[1, 1]
gate_input = state['drift_head.context_gate.0.weight']
# 544 = 256 (z_receiver) + 256 (z_context) + 32 (stage)
gate_components = {
    'Receiver\n(z_recv)': gate_input[:, :256].norm().item(),
    'Context\n(z_ctx)': gate_input[:, 256:512].norm().item(),
    'Stage': gate_input[:, 512:].norm().item(),
}
bars = ax.bar(gate_components.keys(), gate_components.values(), color=['#27ae60', '#c0392b', '#3498db'])
ax.set_ylabel('Weight Norm')
ax.set_title('D. Gate Input Weights')

plt.tight_layout()
plt.savefig('figures/publication/fig_model_weight_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Ablation Study Results

Systematic ablations test three hypotheses: (1) niche context improves transition modeling, (2) dual-reference encoding outperforms single-reference, and (3) the gating mechanism provides essential flexibility. Each configuration modifies a single architectural component while holding others fixed, evaluated across 5-fold cross-validation with 3 random seeds per fold.

In [ ]:
# Load ablation results
with open(RESULTS_DIR / 'comparison_report.json') as f:
    report = json.load(f)

full_loss = report['full_model']['mean_val_loss']
full_std = report['full_model']['std_val_loss']

print('Full Model Performance')
print('=' * 50)
print(f'Validation Loss: {full_loss:.6f} ± {full_std:.6f}')
print(f'N runs: {report["full_model"]["n_runs"]}')

print('\nAblation Results (sorted by impact)')
print('=' * 50)
ablations = sorted(report['ablations'].items(), key=lambda x: x[1]['delta_vs_full'], reverse=True)
for name, data in ablations:
    delta = data['delta_vs_full']
    print(f'{name:25s}: {data["mean_val_loss"]:.6f} ({delta:+.1f}%)')

In [ ]:
# Ablation visualization
fig = plt.figure(figsize=(14, 6))
gs = fig.add_gridspec(1, 2, width_ratios=[2, 1.2], wspace=0.25)

# Panel A: Violin plot of validation losses
ax1 = fig.add_subplot(gs[0])

# Build dataframe
rows = []
for loss in report['full_model']['all_losses']:
    rows.append({'config': 'Full Model', 'loss': loss})
for name, data in ablations:
    label = name.replace('_', ' ').title()
    for loss in data['all_losses']:
        rows.append({'config': label, 'loss': loss})

df = pd.DataFrame(rows)
order = ['Full Model'] + [name.replace('_', ' ').title() for name, _ in ablations]

# Color by delta
palette = {'Full Model': '#27ae60'}
for name, data in ablations:
    label = name.replace('_', ' ').title()
    delta = data['delta_vs_full']
    if delta > 8:
        palette[label] = '#c0392b'
    elif delta > 4:
        palette[label] = '#e74c3c'
    elif delta > 0:
        palette[label] = '#f39c12'
    else:
        palette[label] = '#3498db'

sns.violinplot(data=df, x='config', y='loss', order=order, palette=palette, ax=ax1, inner='box', cut=0)
ax1.axhline(full_loss, color='#27ae60', ls='--', lw=1.5, alpha=0.7, label=f'Full model: {full_loss:.4f}')
ax1.set_xlabel('')
ax1.set_ylabel('Validation Loss (MSE)')
ax1.set_title('A. Ablation Study: Validation Loss Distribution')
ax1.tick_params(axis='x', rotation=45)
ax1.legend(loc='upper right')

# Panel B: Delta bar chart
ax2 = fig.add_subplot(gs[1])
names = [name.replace('_', ' ').title() for name, _ in ablations]
deltas = [data['delta_vs_full'] for _, data in ablations]
colors = [palette[n] for n in names]

y_pos = np.arange(len(names))
bars = ax2.barh(y_pos, deltas, color=colors, edgecolor='white', linewidth=0.5)
ax2.axvline(0, color='black', lw=1)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(names)
ax2.set_xlabel('% Change vs Full Model')
ax2.set_title('B. Component Impact')
ax2.invert_yaxis()

for bar, delta in zip(bars, deltas):
    x_pos = delta + 0.3 if delta >= 0 else delta - 0.3
    ha = 'left' if delta >= 0 else 'right'
    ax2.text(x_pos, bar.get_y() + bar.get_height()/2, f'{delta:+.1f}%',
             va='center', ha=ha, fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/publication/fig_ablation_violin.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Baseline comparison
print('Baseline Comparison')
print('=' * 50)
print(f'{"Model":25s} {"Val Loss":>12s} {"Improvement":>12s}')
print('-' * 50)
print(f'{"StageBridge":25s} {full_loss:12.4f} {"(reference)":>12s}')

for name in ['graphsage', 'pooling', 'deepsets', 'set_transformer']:
    if name in report['baselines']:
        data = report['baselines'][name]
        improvement = data['mean_val_loss'] / full_loss
        print(f'{name.title():25s} {data["mean_val_loss"]:12.4f} {improvement:11.1f}x worse')

## 4. Key Findings

The weight analysis reveals that the trained model relies primarily on the **latent-only pathway** (receiver cell state alone) rather than the context pathway (cross-attention over niche tokens). The latent-only path has approximately 3x higher weight magnitude than the context output projection, indicating that even when context is available, the model learned to down-weight its contribution.

This finding aligns with the ablation results: removing niche context causes only +2.2% degradation, substantially less than removing the gating mechanism (+11.3%) or using single-reference encoding (HLCA-only: +7.2%, LuCA-only: +5.3%).

Importantly, the ring tokens **are** differentiated---PMA seeds show low cosine similarity (0.06-0.22) and input projections are nearly orthogonal. The architecture is capable of learning ring-specific patterns; the data simply does not strongly reward it.

In [ ]:
# Summary statistics
print('Summary of Findings')
print('=' * 60)
print()
print('1. BASELINE COMPARISON')
print(f'   StageBridge achieves {report["baselines"]["graphsage"]["mean_val_loss"]/full_loss:.0f}x lower loss than GraphSAGE')
print(f'   StageBridge achieves {report["baselines"]["pooling"]["mean_val_loss"]/full_loss:.0f}x lower loss than PoolingMLP')
print()
print('2. ABLATION RANKING (by degradation)')
for i, (name, data) in enumerate(ablations[:5], 1):
    print(f'   {i}. {name.replace("_", " ").title()}: {data["delta_vs_full"]:+.1f}%')
print()
print('3. PATHWAY ANALYSIS')
print(f'   Latent-only / Context ratio: {latent_only_norm/context_out_norm:.1f}x')
print(f'   Default gate value: {torch.sigmoid(torch.tensor(gate_bias)).item():.2f}')
print()
print('4. INTERPRETATION')
print('   The gating mechanism is the most critical component.')
print('   Niche context provides modest benefit (+2.2% when removed).')
print('   Dual-reference encoding outperforms single-reference (5-7% gain).')

## 5. Conclusions

This analysis reveals a nuanced picture of niche-conditioned transition modeling. While StageBridge dramatically outperforms baselines (16-22x lower validation loss), the ablation study indicates that the **gating mechanism**---not niche context per se---drives most of the improvement. The gate provides essential flexibility for modeling heterogeneous cell populations where some cells are microenvironment-dependent while others follow cell-autonomous programs.

The modest contribution of explicit niche context (+2.2% when removed) suggests one of three possibilities:

1. **Progression signal is predominantly cell-intrinsic** in this dataset, with receiver cell state sufficient for transition prediction.

2. **Ring pooling dilutes specific interactions**. Mean-pooling neighbors within distance bands may wash out sparse but biologically relevant signals (e.g., one IL1B+ macrophage among many neighbors).

3. **Reference embeddings encode niche information implicitly**. The HLCA and LuCA latent positions may already reflect microenvironmental context through the atlases' training data.

Future work should investigate continuous distance encoding, sparse attention to individual neighbors rather than pooled rings, and explicit ligand-receptor gene programs as additional conditioning signals.

In [ ]:
# Final figure: Combined summary
fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# Panel A: Baseline comparison bar chart
ax = fig.add_subplot(gs[0, 0])
models = ['StageBridge', 'GraphSAGE', 'Pooling', 'DeepSets', 'Set Transformer']
losses = [full_loss] + [report['baselines'].get(m.lower().replace(' ', '_'), {}).get('mean_val_loss', 0) 
                        for m in models[1:]]
colors = ['#27ae60', '#9b59b6', '#7f8c8d', '#3498db', '#e67e22']
bars = ax.bar(models, losses, color=colors)
ax.set_ylabel('Validation Loss')
ax.set_title('A. Baseline Comparison')
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, losses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', fontsize=8)

# Panel B: Ablation impact
ax = fig.add_subplot(gs[0, 1])
top_ablations = ablations[:6]
names = [n.replace('_', ' ').title()[:15] for n, _ in top_ablations]
deltas = [d['delta_vs_full'] for _, d in top_ablations]
colors = ['#c0392b' if d > 5 else '#f39c12' if d > 0 else '#3498db' for d in deltas]
ax.barh(range(len(names)), deltas, color=colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('% Change vs Full')
ax.set_title('B. Top Ablation Impacts')
ax.invert_yaxis()

# Panel C: Pathway weights
ax = fig.add_subplot(gs[0, 2])
ax.bar(['Latent-only', 'Context'], [latent_only_norm, context_out_norm], color=['#27ae60', '#c0392b'])
ax.set_ylabel('Weight Norm')
ax.set_title('C. Drift Head Pathways')
ax.text(0, latent_only_norm + 0.2, f'{latent_only_norm:.1f}', ha='center')
ax.text(1, context_out_norm + 0.2, f'{context_out_norm:.1f}', ha='center')

# Panel D: Ring similarity matrix
ax = fig.add_subplot(gs[1, 0])
sns.heatmap(sim, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=['R1', 'R2', 'R3', 'R4'],
            yticklabels=['R1', 'R2', 'R3', 'R4'], ax=ax)
ax.set_title('D. Ring Differentiation')

# Panel E: Semi-synthetic ground truth
ax = fig.add_subplot(gs[1, 1])
non_int = ~labels['is_interacting'].values
ax.scatter(coords.loc[non_int, 'x'], coords.loc[non_int, 'y'],
           c='#e0e0e0', s=4, alpha=0.3)
ax.scatter(coords.loc[labels['is_interacting'].values, 'x'],
           coords.loc[labels['is_interacting'].values, 'y'],
           c='#c0392b', s=8, alpha=0.6)
ax.set_xlabel('X (μm)')
ax.set_ylabel('Y (μm)')
ax.set_title(f'E. Ground-Truth Interactions\n({labels["is_interacting"].sum()} cells, {100*labels["is_interacting"].mean():.0f}%)')

# Panel F: Key metrics table
ax = fig.add_subplot(gs[1, 2])
ax.axis('off')
table_data = [
    ['Metric', 'Value'],
    ['Full model loss', f'{full_loss:.4f}'],
    ['Best baseline', f'{report["baselines"]["graphsage"]["mean_val_loss"]:.4f}'],
    ['Improvement', f'{report["baselines"]["graphsage"]["mean_val_loss"]/full_loss:.0f}x'],
    ['No-gate impact', '+11.3%'],
    ['No-niche impact', '+2.2%'],
    ['Pathway ratio', f'{latent_only_norm/context_out_norm:.1f}x'],
]
table = ax.table(cellText=table_data, loc='center', cellLoc='left',
                 colWidths=[0.5, 0.4])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)
ax.set_title('F. Summary Metrics')

plt.savefig('figures/publication/fig_combined_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFigures saved to figures/publication/')